GEETA 4B — C6 · Answer-leak guardrail (output side).

Trained behaviour (M3 hint-ladder + refusal data) is the FIRST line of
defence. This guardrail is the SECOND: it parses the model's outgoing text
for the problem's final answer (and obvious reformulations) and blocks +
re-prompts if it appears.

Matching (conservative — better to over-block and re-prompt):
* numbers:  ``42``, ``42.5``, ``1,024``, ``$6.55``, ``6 dollars 55 cents``,
  ``7/8`` fractions, ``+3`` — token-boundary + value equality
* spoken numbers (config match_spoken_numbers): "forty-two", "six dollars
  fifty five cents", "a hundred and two"
* times: ``14:30`` / ``2:30 pm`` compared as minutes of day
* words: whole phrase, case-insensitive (e.g. "ninety eight", "triangle")

Leak = the final answer stated in assistant text that is going out to the
child. If the child ALREADY said/typed the value in the current turn, or it
is part of the question the child is looking at, restating it is not a leak
(mirrors the M3 data-gate semantics, see notebooks/03 cell 10).

Every trigger is logged (config leak_guard.log_triggers) -> dev report.

## Bootstrap

This notebook is a GEETA runtime module. Executed standalone (e.g. `jupyter execute tools/verifier.ipynb`),
the next cell registers the notebook-as-module loader so `geeta.*` imports resolve to `tools/*.ipynb`.
When imported as `geeta.<name>` via `tools/_nbmodule.ipynb`, the same cell runs (idempotent).


In [ ]:
# --- GEETA notebook-module bootstrap (idempotent; makes tools/*.ipynb importable as geeta.*) ---
import json as _json
import pathlib as _pathlib

_ROOT = _pathlib.Path.cwd()
if not (_ROOT / "tools" / "_nbmodule.ipynb").exists():      # run from project root or tools/
    _ROOT = _ROOT.parent
_nb = _json.loads((_ROOT / "tools" / "_nbmodule.ipynb").read_text())
_src = "".join("".join(c["source"]) for c in _nb["cells"] if c["cell_type"] == "code")
exec(compile(_src, str(_ROOT / "tools" / "_nbmodule.ipynb"), "exec"), globals())
bootstrap(_ROOT)  # noqa: E501


In [ ]:

from __future__ import annotations

import json
import re
import time
from dataclasses import dataclass
from fractions import Fraction
from pathlib import Path
from typing import Any, Optional

In [ ]:

# ---------------------------------------------------------------------------
# Answer normalisation (shared with the M3 data gate; exact same semantics)
# ---------------------------------------------------------------------------

_NUM_TOK = re.compile(r"(?<![\d.])\d[\d,]*(?:\.\d+)?")

In [ ]:
_FRAC_TOK = re.compile(r"(?<![\d.])\d+/\d+(?![\d.])")

In [ ]:
_TIME_TOK = re.compile(r"(?<!\d)(\d{1,2}):(\d{2})(?!\d)")

In [ ]:
_DOLLCENTS = re.compile(
    r"(?<!\d)(\d+(?:\.\d+)?)\s*(?:dollars?|bucks)\b\s*(?:and\s+)?(\d+)\s*cents?\b", re.I)

In [ ]:
_AMPM = re.compile(r"\b(am|pm)\b", re.I)

In [ ]:


def parse_answer_value(a: Any) -> tuple[str, Any]:
    """Normalize a final answer to ('num' | 'time' | 'word', value).

    num  -> float value            ( "$6.55" -> 6.55, "1/2" -> 0.5, "1,024" -> 1024 )
    time -> minutes of day         ( "14:30" -> 870, "2:30" -> 150 )
    word -> lowercased phrase      ( "triangle" -> "triangle" )
    """
    s = str(a).strip()
    t = s.replace(",", "").replace("$", "").replace("%", "").replace(" ", "")
    if re.fullmatch(r"\d{1,2}:\d{2}", t):
        h, m = t.split(":")
        return "time", int(h) * 60 + int(m)
    if re.fullmatch(r"-?\d+(/\d+)?", t):
        num, _, den = t.partition("/")
        d = int(den) if den else 1
        return "num", (int(num) / d if d else None)
    if re.fullmatch(r"-?\d+(\.\d+)?", t):
        return "num", float(t)
    # Session 13 fix: word-form numbers are legitimately written either way
    # ("ninety-eight" vs "ninety eight") - normalise hyphens to spaces (and
    # collapse repeats) so a hyphenation choice doesn't register as a wrong
    # answer. Collapse multi-space too, in case of double spaces upstream.
    return "word", re.sub(r"\s+", " ", s.lower().replace("-", " ")).strip()

In [ ]:


# ---------------------------------------------------------------------------
# Number-word parsing (spoken numbers: "forty-two", "a hundred and two")
# ---------------------------------------------------------------------------

_ONES = {"zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5,
         "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10, "eleven": 11,
         "twelve": 12, "thirteen": 13, "fourteen": 14, "fifteen": 15,
         "sixteen": 16, "seventeen": 17, "eighteen": 18, "nineteen": 19}

In [ ]:
_TENS = {"twenty": 20, "thirty": 30, "forty": 40, "fifty": 50, "sixty": 60,
         "seventy": 70, "eighty": 80, "ninety": 90}

In [ ]:
_SCALES = {"hundred": 100, "thousand": 1000, "million": 10**6}

In [ ]:


def _parse_number_word(tokens: list[str]) -> Optional[float]:
    """Parse an English number phrase (< 1e6): "six dollars fifty five cents"
    handled by caller; here: ['a','hundred','and','two'] -> 102.0"""
    toks = [t.lower().strip(" ,-") for t in tokens if t.strip(" ,-")]
    # "a"/"an" is only a number before a scale word ("a hundred"). On its own it
    # is just an article: "Here's a trick" is not the number 1. (2026-10-13: this
    # made every reply with "a" look like it leaked an answer of 1, and let a kid's
    # "a" make the answer 1 count as already "visible" to the guard.)
    if toks and toks[0] in ("a", "an") and (len(toks) == 1 or toks[1] not in _SCALES):
        return None
    total, current = 0, 0
    for t in tokens:
        t = t.lower().strip(" ,-")
        if not t:
            continue
        if t in ("a", "an"):
            current = 1
        elif t in _ONES:
            current += _ONES[t]
        elif t in _TENS:
            current += _TENS[t]
        elif t == "and":
            continue
        elif t in _SCALES:
            if current == 0:
                current = 1
            total += current * _SCALES[t]
            current = 0
        else:
            return None
    return total + current

In [ ]:


_NUMWORDS = (r"a|an|zero|one|two|three|four|five|six|seven|eight|nine|ten|"
             r"eleven|twelve|thirteen|fourteen|fifteen|sixteen|seventeen|eighteen|"
             r"nineteen|twenty|thirty|forty|fifty|sixty|seventy|eighty|ninety|"
             r"hundred|thousand|million")

In [ ]:
_NUMWORD_RUN = re.compile(rf"\b(?:{_NUMWORDS})(?:[\s,\-]+(?:and|{_NUMWORDS}))*\b", re.I)

In [ ]:


def _number_word_chunks(tokens: list[str]) -> list[int]:
    """Split a run of number words into the numbers a person actually said.
    Only real English number grammar joins words: "twenty four" -> 24,
    "a hundred and two" -> 102, "two thousand three hundred" -> 2300.
    Anything else is separate numbers: "four and eight" -> [4, 8],
    "four eight" -> [4, 8]. (2026-11-02: these used to be ADDED, so a kid
    saying "four and eight" was marked correct for an answer of 12.)"""
    toks = [t.lower().strip(" ,-") for t in tokens if t.strip(" ,-")]
    out: list[int] = []
    total = cur = 0
    last = None          # None | ones | teen | tens | hundred | scale | and

    def flush():
        nonlocal total, cur, last
        if last is not None and last != "and":
            out.append(total + cur)
        total = cur = 0
        last = None

    for k, t in enumerate(toks):
        nxt = toks[k + 1] if k + 1 < len(toks) else ""
        if t in ("a", "an"):
            flush()
            if nxt in _SCALES:           # "a hundred"; otherwise just an article
                cur, last = 1, "ones"
        elif t == "and":
            if last in ("hundred", "scale"):
                last = "and"
            else:
                flush()
        elif t in _ONES and 0 < _ONES[t] < 10:
            if last not in (None, "tens", "hundred", "scale", "and"):
                flush()
            cur += _ONES[t]
            last = "ones"
        elif t in _ONES:                 # zero, ten .. nineteen
            if last not in (None, "hundred", "scale", "and"):
                flush()
            cur += _ONES[t]
            last = "teen"
        elif t in _TENS:
            if last not in (None, "hundred", "scale", "and"):
                flush()
            cur += _TENS[t]
            last = "tens"
        elif t == "hundred":
            if last == "ones" and cur < 10:
                cur *= 100
            else:
                flush()
                cur = 100
            last = "hundred"
        elif t in _SCALES:               # thousand, million
            if last in ("ones", "teen", "tens", "hundred") and cur:
                total += cur * _SCALES[t]
                cur = 0
            else:
                flush()
                total = _SCALES[t]
            last = "scale"
        else:
            flush()
    flush()
    return out


def spoken_numbers_in_text(text: str) -> list[float]:
    """All number-word phrases in text, parsed to floats
    ("forty-two" -> 42, "a hundred and two" -> 102, "four and eight" -> 4, 8)."""
    out: list[float] = []
    for m in _NUMWORD_RUN.finditer(text):
        toks = re.split(r"[\s,\-]+", m.group(0))
        out += [float(v) for v in _number_word_chunks(toks)]
    return out


# "6 million" / "24 thousand": digits with a scale word after them
_DIGIT_SCALE = re.compile(r"(?<![\d.])(\d+(?:\.\d+)?)\s*(hundred|thousand|million)\b", re.I)


def scaled_numbers_in_text(text: str) -> list[float]:
    return [float(n) * _SCALES[s.lower()] for n, s in _DIGIT_SCALE.findall(text or "")]

In [ ]:


# ---------------------------------------------------------------------------
# Text scanning
# ---------------------------------------------------------------------------

def answer_in_text(kind: str, val: Any, raw: str, text: str) -> bool:
    """True if the final-answer value is stated anywhere in ``text``.

    Mirrors the M3 assembly gate (notebooks/03 cell 10) with spoken-number
    support added for the runtime guardrail.
    """
    if not text:
        return False
    phrase_tok = val if kind == "word" else str(raw).strip()
    if phrase_tok and re.search(r"(?<!\w)" + re.escape(phrase_tok) + r"(?!\w)",
                                text, flags=re.I):
        return True
    if val is None:
        return False
    if kind == "time":
        for h, mm in _TIME_TOK.findall(text):
            if int(h) * 60 + int(mm) == val:
                return True
        return False
    if kind == "word":
        return False  # value-form matching only for numeric/time answers
    vals = [float(x.replace(",", "")) for x in _NUM_TOK.findall(text)]
    for f in _FRAC_TOK.findall(text):
        n, d = f.split("/")
        if int(d):
            vals.append(int(n) / int(d))
    vals += [float(d) + int(c) / 100 for d, c in _DOLLCENTS.findall(text)]
    vals += spoken_numbers_in_text(text)
    vals += scaled_numbers_in_text(text)
    return any(abs(v - val) < 1e-9 for v in vals)

In [ ]:


# ---------------------------------------------------------------------------
# Guardrail
# ---------------------------------------------------------------------------

@dataclass
class GuardResult:
    blocked: bool
    leak_found: Optional[str]
    action: str          # "pass" | "block" | "reprompt"
    outgoing_text: str   # what actually goes out (replacement if blocked)
    reason: str = ""

    def to_dict(self) -> dict:
        return {
            "blocked": self.blocked,
            "leak_found": self.leak_found,
            "action": self.action,
            "outgoing_text": self.outgoing_text,
            "reason": self.reason,
        }

In [ ]:


class LeakGuard:
    """Output guardrail for one problem.

    ``expected_answer`` — the problem's final answer (any string/number).
    ``child_already_knows`` — text the child can already see/said (question
    as displayed to the child + child's current-turn message). Values that
    appear there are NOT leaks when restated.
    """

    def __init__(self, expected_answer: Any, *,
                 child_visible_text: str = "",
                 reprompt_message: str = ("I love that you're keen to find out! "
                                          "But the fun is working it out yourself. "
                                          "Let's try the first little step together."),
                 block_and_reprompt: bool = True,
                 match_spoken_numbers: bool = True,
                 match_reformulations: bool = True):
        self.raw_answer = str(expected_answer).strip()
        self.kind, self.val = parse_answer_value(self.raw_answer)
        self.child_visible_text = child_visible_text
        self.reprompt_message = reprompt_message
        self.block_and_reprompt = block_and_reprompt
        self.match_spoken_numbers = match_spoken_numbers
        self.match_reformulations = match_reformulations

    def check(self, assistant_text: str,
              child_current_text: str = "") -> GuardResult:
        t0 = time.perf_counter()
        text = assistant_text or ""
        # what the child can currently see/said (their turn counts as known)
        visible = self.child_visible_text
        if child_current_text:
            visible = f"{visible} {child_current_text}"
        leak = self._scan(text, visible)
        if leak is None:
            return GuardResult(False, None, "pass", text,
                               f"{(time.perf_counter()-t0)*1000:.1f}ms")
        if self.block_and_reprompt:
            return GuardResult(True, leak, "reprompt", self.reprompt_message,
                               f"value {leak!r} in outgoing text")
        return GuardResult(True, leak, "block", "", f"value {leak!r} in outgoing text")

    def _scan(self, text: str, child_visible: str) -> Optional[str]:
        """Return the leaking value-form if the final answer appears in text
        in a way the child has NOT already seen/said."""
        if self.kind == "num" and self.val is not None:
            return self._scan_numeric(text, child_visible)
        if self.kind == "time" and self.val is not None:
            if answer_in_text("time", self.val, self.raw_answer, text) and \
               not self._child_states(visible=child_visible):
                return f"time {self.raw_answer}"
            return None
        if self.kind == "word":
            if re.search(r"(?<!\w)" + re.escape(str(self.val)) + r"(?!\w)",
                         text, flags=re.I) and \
               not self._child_states(visible=child_visible):
                return str(self.val)
            return None
        return None

    def _child_states(self, visible: str = "") -> bool:
        src = visible or self.child_visible_text
        return answer_in_text(self.kind, self.val, self.raw_answer, src) if src else False

    def _scan_numeric(self, text: str, child_visible: str) -> Optional[str]:
        if not self.match_reformulations:
            return self.raw_answer if self.raw_answer in text else None
        if answer_in_text("num", self.val, self.raw_answer, text):
            if self._child_states(visible=child_visible):
                return None
            # find the human form to report
            for tok in _NUM_TOK.findall(text):
                if abs(float(tok.replace(",", "")) - self.val) < 1e-9:
                    return tok
            for f in _FRAC_TOK.findall(text):
                n, d = f.split("/")
                if int(d) and abs(int(n) / int(d) - self.val) < 1e-9:
                    return f
            if self.match_spoken_numbers:
                for v in spoken_numbers_in_text(text):
                    if abs(v - self.val) < 1e-9:
                        return f"<spoken {self.val:g}>"
            return self.raw_answer
        return None

In [ ]:


class GuardLog:
    """Append-only JSONL trigger log -> dev report + scorecard."""

    def __init__(self, path: str | Path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.path.touch(exist_ok=True)

    def record(self, res: GuardResult, meta: dict | None = None) -> None:
        row = {"ts": time.strftime("%Y-%m-%dT%H:%M:%S%z"), **(meta or {})}
        row.update(res.to_dict())
        with self.path.open("a") as f:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    def read(self) -> list[dict]:
        if not self.path.exists():
            return []
        return [json.loads(l) for l in self.path.read_text().splitlines() if l.strip()]